# 01 — Checks before the analysis

Everything in this notebook is allowed BEFORE the `prereg` tag: arm counts, covariates, and
outcomes **pooled** with arms hidden. Rule for this notebook: **never group an outcome
(`visit`, `conversion`, `spend`) by `segment`.**

Fill in every `___`. Hints are in the comments. A cell with a `___` left in it will error — that is expected until you fill it.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats

In [2]:
df = pd.read_csv("../data/raw/hillstrom.csv")

## 1. Set the outcomes aside

Make a dataframe without the outcome columns. Use `covariates` (not `df`) for anything split by arm.

In [3]:
outcomes = ['visit', 'conversion', 'spend']          # the three outcome column names, as strings
covariates = df.drop(columns=outcomes)
covariates.head()                   # check: 9 columns, no outcomes

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail


## 2. First look (pooled — safe)

In [16]:
print(df.shape)          # (rows, columns) -> expect (64000, 12)
print(df.dtypes)          # type of each column
print(df.isna().sum())  # missing values per column

(64000, 12)
recency              int64
history_segment        str
history            float64
mens                 int64
womens               int64
zip_code               str
newbie               int64
channel                str
segment                str
visit                int64
conversion           int64
spend              float64
dtype: object
recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64


In [17]:
# value_counts() lists each unique value and how often it appears.
# Run it for zip_code, channel and history_segment.
#covariates["zip_code"].value_counts()
#covariates["channel"].value_counts()
covariates["history_segment"].value_counts()

history_segment
1) $0 - $100        22970
2) $100 - $200      14254
3) $200 - $350      12289
4) $350 - $500       6409
5) $500 - $750       4911
6) $750 - $1,000     1859
7) $1,000 +          1308
Name: count, dtype: int64

**Notes on the data:**
- No missing values in any column.
- `zip_code`: Suburban is the largest group, Urban just below, Rural much smaller. **The raw file spells Suburban as `Surburban`** — noted, left as is so the data matches the source.
- `channel`: Web and Phone are nearly equal (about 200 apart); Multichannel is much smaller.
- `history_segment`: 7 bands of past-year spend, from $0–100 (the largest, about a third of customers) up to $1,000+ (the smallest).

## 3. Sample ratio mismatch (SRM)

Question: did each arm get the share it was supposed to? The design intended thirds.

In [14]:
observed = covariates["segment"].value_counts()   # customers per arm
expected = [len(df) / 3] * 3                # what a perfect 1/3 split gives
print(observed)
print(expected)

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64
[21333.333333333332, 21333.333333333332, 21333.333333333332]


In [15]:
# scipy.stats.chisquare(f_obs=..., f_exp=...) returns (statistic, p-value)
stat, p = scipy.stats.chisquare(f_obs=observed, f_exp=expected)
print(f"chi-square = {stat:.3f}, p = {p:.4f}")

chi-square = 0.203, p = 0.9037


**Interpretation:**
The split matches the intended thirds (chi-square 0.20, p = 0.90), so there's no evidence of sample ratio mismatch.

**Why do SRM checks flag at p < 0.001 rather than 0.05?**
Because the check is run on every experiment, often every day while it runs. At 0.05, one healthy experiment in twenty would be flagged by chance, and each false alarm sends someone hunting for a bug that isn't there. A real SRM is usually caused by a bug that shifts thousands of users, which gives a p-value far below 0.001 anyway, so the stricter threshold costs almost nothing in missed problems.

**If SRM failed, what would it tell you, and would you still analyze the results?**
It would mean customers were lost or added unevenly between arms, so the arms may no longer contain the same kind of customer, and any difference in spend could come from that rather than from the email. I would not trust the results until I found the cause (an assignment bug, emails that bounced and were dropped, filtering applied to one arm). Randomly cutting the bigger arm down to size does not fix it: it equalizes the counts, but the customers missing from the smaller arm were not missing at random, so the bias stays.

## 4. Covariate balance

If randomization worked, customers in each arm should look alike BEFORE the email.
Measure it with the **standardized mean difference (SMD)** for each covariate, each email arm vs control:

SMD = (mean_treatment − mean_control) / sqrt((var_treatment + var_control) / 2)

Rule of thumb: |SMD| < 0.1 is balanced.

Categorical columns need to become 0/1 columns first — `pd.get_dummies` does that.

In [27]:
# Turn the categorical columns into 0/1 columns; keep segment as it is.
cat_cols = ['history_segment', 'zip_code', 'channel']                  # the text columns that are covariates (not segment)
X = pd.get_dummies(covariates, columns=cat_cols, dtype=int)
X.head()

,recency,history,mens,womens,newbie,segment,history_segment_1) $0 - $100,history_segment_2) $100 - $200,history_segment_3) $200 - $350,history_segment_4) $350 - $500,history_segment_5) $500 - $750,"history_segment_6) $750 - $1,000","history_segment_7) $1,000 +",zip_code_Rural,zip_code_Surburban,zip_code_Urban,channel_Multichannel,channel_Phone,channel_Web
0,10,142.44,1,0,0,Womens E-Mail,0,1,0,0,0,0,0,0,1,0,0,1,0
1,6,329.08,1,1,1,No E-Mail,0,0,1,0,0,0,0,1,0,0,0,0,1
2,7,180.65,0,1,1,Womens E-Mail,0,1,0,0,0,0,0,0,1,0,0,0,1
3,9,675.83,1,0,1,Mens E-Mail,0,0,0,0,1,0,0,1,0,0,0,0,1
4,2,45.34,1,0,0,Womens E-Mail,1,0,0,0,0,0,0,0,0,1,0,0,1


In [26]:
def smd(a, b):
    # a, b: two pandas Series (one column, two arms). Return the SMD from the formula above.
    # Hint: a.mean(), a.var()
    return(a.mean() - b.mean())/np.sqrt(((a.var()+b.var())/2))

In [28]:
control = X[X["segment"] == "No E-Mail"]
rows = []
for arm in ["Mens E-Mail", "Womens E-Mail"]:
    treated = X[X["segment"] == arm]
    for col in X.columns.drop("segment"):
        rows.append({"arm": arm, "covariate": col, "smd": smd(treated[col], control[col])})
balance = pd.DataFrame(rows)
balance.sort_values("smd", key=abs, ascending=False).head(10)   # largest imbalances first

,arm,covariate,smd
6,Mens E-Mail,history_segment_2) $100 - $200,-0.016360
11,Mens E-Mail,"history_segment_7) $1,000 +",0.015835
24,Womens E-Mail,history_segment_2) $100 - $200,-0.014286
12,Mens E-Mail,zip_code_Rural,0.013659
13,Mens E-Mail,zip_code_Surburban,-0.011743
25,Womens E-Mail,history_segment_3) $200 - $350,0.011351
17,Mens E-Mail,channel_Web,0.011014
5,Mens E-Mail,history_segment_1) $0 - $100,0.010917
9,Mens E-Mail,history_segment_5) $500 - $750,-0.009740
28,Womens E-Mail,"history_segment_6) $750 - $1,000",-0.008819


**Interpretation:** *(is any |SMD| above 0.1? what does that say about the randomization?)*
No covariate is imbalanced: the largest |SMD| is 0.016, far below 0.1. The arms look alike before the email, which is consistent with the randomization having worked.

## 5. Spend, pooled with arms hidden

This sets your **heavy-tail rule** for the pre-registration. Use `df["spend"]` on its own —
no `segment` anywhere in these cells.

In [24]:
spend = df["spend"]
print("share of customers with zero spend:", (spend == 0).mean())
print(spend.quantile([0.90, 0.99, 0.999]))     # where would a percentile cap land?

share of customers with zero spend: 0.99096875
0.900      0.00000
0.990      0.00000
0.999    243.66049
Name: spend, dtype: float64


In [25]:
# Among purchasers only (spend > 0): how spread out is it?
buyers = spend[spend > 0]
print(len(buyers))
print(buyers.describe())
print(buyers.quantile([0.95, 0.99]))

578
count    578.000000
mean     116.363547
std      107.871537
min       29.990000
25%       32.272500
50%       80.795000
75%      153.350000
max      499.000000
Name: spend, dtype: float64
0.95    362.2
0.99    499.0
Name: spend, dtype: float64


**What this means for the heavy-tail rule:** *(would a 99th-percentile cap on ALL customers do anything?
what rule will you pre-register instead?)*
99.1% of customers spent $0, so the 99th percentile is $0 and a 99th-percentile cap would erase every purchase. Among the 578 buyers, spend runs from $29.99 to $499, with 12 buyers at exactly $499, so the data looks top-coded at $499 before release. Because the tail is already bounded, I will not add a cap; I will analyze raw spend and use a bootstrap confidence interval as the robustness check. Limitation: the $499 ceiling hides how much the biggest spenders actually spent, which may understate an effect concentrated among them.

## 6. Blinded minimum detectable effect (MDE)

With ~21,000 customers per arm, what is the smallest effect on spend you could reliably detect?
Use the **pooled** standard deviation of spend (arms hidden).

MDE = (z_alpha + z_power) × sqrt(2 × sd² / n_per_arm)

- z_alpha: two-sided, so use `scipy.stats.norm.ppf(1 - alpha / 2)`
- z_power: for 80% power, `scipy.stats.norm.ppf(0.80)`
- Think: with Holm across two comparisons, the strictest step tests at alpha / 2. Which alpha should you plug in to be safe?

In [ ]:
alpha = 0.025
n_per_arm = observed.min()          # use the smallest arm, to be conservative
sd = df["spend"].std()

z_alpha = scipy.stats.norm.ppf(1-alpha/2)
z_power = scipy.stats.norm.ppf(0.80)
mde = (z_alpha + z_power) * np.sqrt((2*sd*sd)/n_per_arm)
print(f"MDE on spend per customer: ${mde:.3f}")
print(f"as a share of mean spend: {mde / df['spend'].mean():.1%}")

MDE on spend per customer: $0.449
as a share of mean spend: 42.7%


In [33]:
# Same idea for conversion (a proportion): the variance is p * (1 - p), with p the pooled conversion rate.
p_conv = df["conversion"].mean()
mde_conv = (z_alpha + z_power) * np.sqrt((2 * (p_conv * (1-p_conv))) / n_per_arm)
print(f"MDE on conversion rate: {mde_conv:.4f} (pooled rate {p_conv:.4f})")

MDE on conversion rate: 0.0028 (pooled rate 0.0090)


**Interpretation:** *(is that MDE small enough to catch an effect worth acting on? write the number into the
pre-registration's MDE section)*
The smallest effect we could reliably detect on spend is $0.45 per customer, about 43% of mean spend. And on conversion about 0.28 percentage points, about 31% of the conversion rate". A non-significant result would not show the email has no effect; it would mean the test could not distinguish the effect from zero. I will report the confidence interval to show which effect sizes the data rule out. I will add visit as a secondary metric because because visits are far more common than purchases, so the test can detect much smaller changes in them.. 